In [124]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [129]:
class Value:
    def __init__(self, data, _children = (), _op = ''):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._backward = lambda : None
        self._op = _op
    def __repr__(self):
        return f"Value(data: {self.data})"
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = backward
        return out

    def __radd__(self, other):
        return self + other
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad 
        out._backward = backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self, ), '**')
        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out
    
        
    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return -1 * self
        
    def __sub__(self, other):
        return self + (-other)

    def exp(self):
        x = math.exp(self.data)
        out = Value(x, (self, ), 'exp')
        def _backward():
            self.grad += x * out.grad
        out._backward = _backward
        return out
        
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        def backward():
            self.grad += (1 - t**2) * out.grad # we need to accumulate gradients bcoz in the case where the same value object is repeated
        out._backward = backward               # the gradient is overwritten instead of being updated correctly.
        return out
    

    def backward(self):
        self.grad = 1.0
        topo = []
        visited = set()
        def build_topo(v):
            visited.add(v)
            for child in v._prev:
                if child not in visited:
                    build_topo(child)
            topo.append(v)
        build_topo(self)
        for node in reversed(topo):
            node._backward()
        

In [130]:
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
x1w1 = x1*w1; x2w2 = x2*w2
x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b
o = ((2*n).exp() - 1)/((2*n).exp() + 1)
o.grad = 1.0 # important to set this since otherwise all nodes will become 0
o.backward()

In [131]:
print(x1.grad, ' ', w1.grad, ' ', x2.grad, ' ', w2.grad)

-1.5   1.0   0.5   0.0


In [132]:
import torch

In [133]:
torch.Tensor([2.0]).double().dtype

torch.float64

In [134]:
x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True # need to typecast to double since python works on float64 but pytorch is float32
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print(x1.grad.item(), ' ', w1.grad.item(), ' ', x2.grad.item(), ' ', w2.grad.item())

0.7071066904050358
-1.5000003851533106   1.0000002567688737   0.5000001283844369   0.0


In [187]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh();
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        params = []
        for neuron in self.neurons:
            ps = neuron.parameters()
            params.extend(ps)
        return params

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            ps = layer.parameters()
            params.extend(ps)
        return params

In [279]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

Value(data: -0.3622986288159825)

In [280]:
# tiny dataset
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [281]:
#gradient descent
for k in range(30):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - Value(ygt))**2 for yout, ygt in zip(ypred, ys))

    #backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    # update the parameter values
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)

# critical mistake: each time while training in backward pass the grad gets +=. When we start a backward pass, we need to update all grads to 0.0. 
# Here it continues with the previous value that it has.

0 3.5468488581214297
1 0.6041230367586126
2 0.1632159280118796
3 0.11214119849255526
4 0.08922736391162434
5 0.07546596010964918
6 0.06594191471772066
7 0.05879625502352638
8 0.05315793784597071
9 0.0485560578393061
10 0.044708595334722497
11 0.041433372062051964
12 0.038605784088340665
13 0.036136779821816095
14 0.033960490843296075
15 0.032026843539382045
16 0.030296915594505865
17 0.028739891421123293
18 0.02733099393374238
19 0.026050036959587103
20 0.02488038600844655
21 0.023808195823649028
22 0.022821840411292482
23 0.02191147994506877
24 0.021068726926887868
25 0.02028638557179755
26 0.019558246042185146
27 0.018878920329042544
28 0.018243710143335645
29 0.01764849968228683


In [282]:
ypred

[Value(data: 0.9170111921214371),
 Value(data: -0.9652241470369963),
 Value(data: -0.9232854159431795),
 Value(data: 0.9394453132047867)]